In [16]:
import polars as pl

# 1. Load the dataset
df = pl.read_csv("../../Data/final.csv")

# Update schema
edu_schema = pl.List(pl.Struct({
    "university": pl.String,
    "degree_standardized": pl.String,
    "start_date": pl.String,
    "end_date": pl.String
}))

target_universities = [
    "BUV", "British University Vietnam", "Swinburne", "FUV", 
    "Fulbright University Vietnam", "RMIT", "Royal Melbourne Institute of Technology Vietnam",
    "VinUniversity"
]
pattern = "|".join([uni.lower() for uni in target_universities])
target_years = ["2027-01-01", "2026-01-01", "2025-01-01", "2024-01-01"]

# The degrees that disqualify a candidate if completed BEFORE the target degree
advanced_degrees = ["bachelor", "master", "phd", "architect", "engineer"]

# 2. Decode JSON and apply native list evaluations
matching_ids = (
    df.select(["id", pl.col("educations").str.json_decode(edu_schema).alias("edu_structs")])
    .with_columns([
        # Mask 1: Identify the target degree
        pl.col("edu_structs").list.eval(
            pl.element().struct.field("university").str.to_lowercase().str.contains(pattern) &
            (pl.element().struct.field("degree_standardized") == "bachelor") &
            pl.element().struct.field("end_date").is_in(target_years)
        ).alias("target_mask"),
        
        # Array 1: Extract end dates ONLY for advanced/bachelor degrees. 
        # "college" and "high school" will become null and get ignored.
        pl.col("edu_structs").list.eval(
            pl.when(pl.element().struct.field("degree_standardized").is_in(advanced_degrees))
            .then(pl.element().struct.field("end_date").str.strptime(pl.Date, "%Y-%m-%d", strict=False))
            .otherwise(None)
        ).alias("advanced_end_dates"),
        
        # Array 2: Extract all start dates
        pl.col("edu_structs").list.eval(
            pl.element().struct.field("start_date").str.strptime(pl.Date, "%b %Y", strict=False)
        ).alias("all_start_dates")
    ])
    .with_columns(
        # Extract the start date of the specific target degree
        pl.col("all_start_dates").list.get(pl.col("target_mask").list.arg_max()).alias("target_start_date")
    )
    .filter(
        # Condition A: Must have the target degree
        pl.col("target_mask").list.any() & 
        
        # Condition B: The earliest end_date of any ADVANCED degree must not be before the target start_date.
        # This completely ignores prior college/high school degrees.
        (
            pl.col("advanced_end_dates").list.drop_nulls().list.min() >= pl.col("target_start_date")
        ).fill_null(True) 
    )
    .select("id")
    .unique()
)

# 3. Filter the original DataFrame and export to CSV
filtered_df = df.join(matching_ids, on="id", how="inner")
filtered_df.write_csv("fresh-grad-filter-new.csv")